What do we need?

1. Text Spliting by chunking -> LLMs have context window, which limits the amount of words it can process in a single time (in order of thousands).

2. Embedding -> the way LLM convert your prompt to high dimensionality vectors

3. Vector databases -> store the embedded chunks. Ex: InMemory, FAISS, ChromaDB for small/medium projects

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.agents import create_agent
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool

import os
from dotenv import load_dotenv

load_dotenv(override=True) #override to get changes in the .env file

API_KEY = os.getenv('API_KEY')

In [2]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-lite", # smallest model
    google_api_key=API_KEY,
)

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_text(open("curriculum.txt", encoding="utf-8").read())

In [5]:
emb = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=API_KEY
    )

In [6]:
vector_store = InMemoryVectorStore.from_texts(texts=chunks ,embedding=emb)
retriever = vector_store.as_retriever()

In [29]:
@tool()
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_chunks = vector_store.similarity_search(query, k=5)

    return retrieved_chunks

tools = [retrieve_context]

In [14]:
agent = create_agent(
    model=llm,
    system_prompt="You have access to a tool that retrieves information about Bruno's curriculum.",
    tools=tools
)

In [15]:
output = agent.invoke({"messages": [{"role": "user", "content": "What is Bruno's Education?"}]})

In [21]:
output["messages"][-1].content

'Bruno is a Computer Science undergraduate student at the Federal University of Rio de Janeiro (UFRJ), with an expected graduation in 2027. He also did an exchange semester at the Technical University of Munich.'

In [24]:
def response(agent, query):
    return (agent.invoke({"messages": [{"role": "user", "content": query}]}))["messages"][-1].content

In [25]:
response(agent, "Give me a short description of Bruno's projects.")

"I am sorry, I cannot provide a description of Bruno's projects. I can only retrieve information about Bruno's curriculum."

In [26]:
response(agent, "Give me a short description of Bruno's projects that are inside his curriculum text.")

'Bruno has been a Back-end Developer at UFRJ’s Junior Consulting and Microinformatics Company, where he contributed to a full project, gaining experience across the entire software development lifecycle. He also participated in the Extension Project – InformAÇÃO, where he developed training material on Critical Data Literacy for K-12 teachers.'

Trying a smarter model

In [30]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
)

agent = create_agent(
    model=llm,
    system_prompt="You have access to a tool that retrieves information about Bruno's curriculum.",
    tools=tools
)

In [32]:
res = response(agent, "Give me a short description of Bruno's projects.")

In [36]:
res[0]["text"]

'Bruno has worked on several projects, including an Extension Project called "InformAÇÃO" where he developed training material on Critical Data Literacy for K-12 teachers. He also completed a "Retail Store Demand Forecasting Model" as a final project for a Data Science Bootcamp, involving data collection, preprocessing, exploratory analysis, feature engineering, and machine learning model development. Additionally, Bruno developed a "Database-Integrated Website" using Python, SQLAlchemy for a MySQL database, and the Flask framework for the website.'